# Testing PostHoc explanations for the E3 Theia dataset

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
import torch
from torch_geometric.data import Data
import os
import torch.nn.functional as F
import orjson as json
import warnings
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
warnings.filterwarnings('ignore')
from torch_geometric.loader import NeighborLoader
import gc

import multiprocessing

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
%matplotlib inline

In [ ]:
import gdown
urls = ["https://drive.google.com/file/d/1zbgWJgF7F0fI6JhViqQZoo6AWdoV5YFK/view?usp=drive_link",
       "https://drive.google.com/file/d/1dWJecuLXZMksKAPo8348Q6L5DiccsS1u/view?usp=drive_link",
       "https://drive.google.com/file/d/1Kadc6CUTb4opVSDE4x6RFFnEy0P1cRp0/view?usp=drive_link",
       "https://drive.google.com/file/d/10cecNtR3VsHfV0N-gNEeoVeB89kCnse5/view?usp=drive_link"]
for url in urls:
    gdown.download(url, quiet=False, use_cookies=False, fuzzy=True)

In [ ]:
#!rm raw_gz_files/*.gz

In [ ]:
import pandas as pd
import numpy as np  
import os
import json
import tarfile
import argparse
import re

def extract_json_from_tar_gz(folder_path):
    """
    Extract all .json.tar.gz files under 'folder_path'.
    """
    for filename in os.listdir(folder_path):
        file_path = os.path.join(folder_path, filename)

        if filename.endswith('.json.tar.gz'):
            try:
                with tarfile.open(file_path, 'r:gz') as tar:
                    tar.extractall(path=folder_path)
            except Exception as e:
                print(f"Failed to extract {filename}: {str(e)}")
        else:
            pass


def set_username(path):
    global uid, uname

    with open(path, 'r', encoding='utf-8') as file:
        content = file.read()

    user_id_match = re.search(r'"userId":"([^"]+)"', content)
    username_match = re.search(r'"username":\{"string":"([^"]+)"\}', content)

    user_id = user_id_match.group(1) if user_id_match else None
    username = username_match.group(1) if username_match else None
    
    if user_id is not None:
        uid = user_id
    
    if username is not None:
        uname = username

def json_file_iterator(folder_path):
    """
    Yields each JSON record (line) from all .json or .json.* files 
    under 'folder_path', sorted by filename.
    """
    global cdm

    for file_name in sorted(os.listdir(folder_path)):
        if file_name.endswith('.json') or '.json.' in file_name:
            file_path = os.path.join(folder_path, file_name)
            print("Reading File: " + file_path)
            set_username(file_path)
            try:
                with open(file_path, 'r', encoding='utf-8') as file:
                    for line in file:
                        if f"{cdm}.FileObject" in line or f"{cdm}.Event" in line or f"{cdm}.Subject" in line or f"{cdm}.NetFlowObject" in line:
                            if not ("FILE_OBJECT_UNIX_SOCKET" in line or "FILE_OBJECT_DIR" in line):
                                try:
                                    record = json.loads(line.strip())
                                    yield record
                                except json.JSONDecodeError as e:
                                    print(f"Error decoding JSON from {file_name}: {e}")
            except FileNotFoundError:
                print(f"File not found: {file_path}")
            except IOError as e:
                print(f"Error opening file {file_path}: {e}")


def prepare_id_files(folder_path):
    """
    Iterates over all JSON lines in 'folder_path' and writes two files:
      1. *_id_to_type.json
      2. *_net2prop.json
    into a subdirectory {folder_path}/{title}_data.
    """
    global title, scene, cdm, output_dir

    id_to_type_file = f'{output_dir}/{scene}_id_to_type.json'
    net2prop_file   = f'{output_dir}/{scene}_net2prop.json'

    os.makedirs(f'{output_dir}', exist_ok=True) 

    net2prop_buffer = []
    id_to_type_buffer = []

    def append_to_file(file_path, data):
        with open(file_path, 'w') as file:
            for item in data:
                file.write(json.dumps(item) + '\n')

    for line in json_file_iterator(folder_path):
        str_line = json.dumps(line)
        
        if f"avro.cdm{cdm}.NetFlowObject" in str_line:
            net_flow_object = line['datum'][f'com.bbn.tc.schema.avro.cdm{cdm}.NetFlowObject']
            try:
                nodeid = net_flow_object['uuid']
                srcaddr = net_flow_object.get('localAddress')
                srcport = net_flow_object.get('localPort')
                dstaddr = net_flow_object.get('remoteAddress')
                dstport = net_flow_object.get('remotePort')
                
                net2prop_data = {nodeid: [srcaddr, srcport, dstaddr, dstport]}
                id_to_type_data = {nodeid: 'NetFlowObject'}
                net2prop_buffer.append(net2prop_data)
                id_to_type_buffer.append(id_to_type_data)
            except:
                pass

        if f"schema.avro.cdm{cdm}.Subject" in str_line:
            subject = line['datum'][f'com.bbn.tc.schema.avro.cdm{cdm}.Subject']
            uuid = subject['uuid']
            record_type = subject['type'] 
            id_to_type_data = {uuid: record_type}
            id_to_type_buffer.append(id_to_type_data)

        if f"schema.avro.cdm{cdm}.FileObject" in str_line:
            file_object = line['datum'][f'com.bbn.tc.schema.avro.cdm{cdm}.FileObject']
            uuid = file_object['uuid']
            file_type = file_object['type']
            id_to_type_data = {uuid: file_type}
            id_to_type_buffer.append(id_to_type_data)

    append_to_file(net2prop_file, net2prop_buffer)
    append_to_file(id_to_type_file, id_to_type_buffer)


def load_dict_from_jsonl(file_path):
    """
    Loads line-delimited JSON from 'file_path' into a single dict (merged).
    """
    result = {}
    with open(file_path, 'r') as file:
        for line in file:
            data = json.loads(line)
            result.update(data)
    return result


def fill_missing_paths_for_subject_process(df):
    actorID_exec_map = dict(
        df.loc[df['exec'] != '', ['actorID', 'exec']]
          .drop_duplicates(subset=['actorID'])  
          .values
    )
    
    mask = (df['object'] == 'SUBJECT_PROCESS') & (df['properties'] == '')
    rows_to_update = df[mask].index

    for idx in rows_to_update:
        object_id = df.at[idx, 'objectID']
        if object_id in actorID_exec_map:
            df.at[idx, 'properties'] = {'objectpath':actorID_exec_map[object_id]}

    return df

def stitch(data_buffer):
    """
    Uses the ID-to-type and net2prop files in {folder_path}/{title}_data 
    to add object type and netflow attributes to 'data_buffer'.
    """
    global title, scene, cdm, output_dir

    id_to_type_file = f'{output_dir}/{scene}_id_to_type.json'
    net2prop_file   = f'{output_dir}/{scene}_net2prop.json'
    
    id_to_type = load_dict_from_jsonl(id_to_type_file)
    net2prop   = load_dict_from_jsonl(net2prop_file)
    info       = data_buffer
    
    for i in range(len(info)):
        try:
            typ = id_to_type[info[i]['objectID']]
            info[i]['object'] = typ
            info[i]['actor_type'] = id_to_type[info[i]['actorID']]
            if typ == 'NetFlowObject':
                attr = net2prop[info[i]['objectID']]
                info[i]['properties'] = {
                        'localAddress': attr[0],
                        'localPort': str(attr[1]),
                        'remoteAddress': attr[2],
                        'remotePort': str(attr[3])
                    }
        except:
            info[i]['object'] = None
            info[i]['actor_type'] = None
            
    df = pd.DataFrame.from_records(info)
    df = df.dropna()

    os.remove(id_to_type_file)
    os.remove(net2prop_file)

    output_file = os.path.join(output_dir, f"{title}_{scene}.json")
    df = fill_missing_paths_for_subject_process(df)
    df.to_json(output_file, orient='records', lines=True)  


def query_json(folder_path):
    global title, scene, cdm, uid, uname

    info_buffer = []

    for line in json_file_iterator(folder_path):
        try:
            action = line['datum'][f'com.bbn.tc.schema.avro.cdm{cdm}.Event']['type']
        except:
            action = ''
        try:
            hostid = line['hostId']
        except:
            hostid = ''
        try:
            actor = line['datum'][f'com.bbn.tc.schema.avro.cdm{cdm}.Event']['subject'][f'com.bbn.tc.schema.avro.cdm{cdm}.UUID']
        except:
            actor = ''
        try:
            obj = line['datum'][f'com.bbn.tc.schema.avro.cdm{cdm}.Event']['predicateObject'][f'com.bbn.tc.schema.avro.cdm{cdm}.UUID']
        except:
            obj = ''
        try:
            cmd = line['datum'][f'com.bbn.tc.schema.avro.cdm{cdm}.Event']['properties']['map']['exec']
        except:
            cmd = ''
        try:
            path = {'objectpath': line['datum'][f'com.bbn.tc.schema.avro.cdm{cdm}.Event']['predicateObjectPath']['string']}
        except:
            path = ''
        try:
            timestampnano = line['datum'][f'com.bbn.tc.schema.avro.cdm{cdm}.Event']['timestampNanos']
        except:
            timestampnano = ''
        try:
            timestamp = line['@timestamp']
        except:
            timestamp = ''
        try:
            cmdline = line['datum'][f'com.bbn.tc.schema.avro.cdm{cdm}.Event']['properties']['map']['cmdLine']
        except:
            cmdline = ''

        info_data = {
            'actorID': actor,
            'objectID': obj,
            'action': action,
            'timestampNanos': timestampnano,
            'timestamp': timestamp,
            'exec': cmd,
            'properties': path,
            'cmdline': cmdline,
            'hostid': hostid,
        }
        info_buffer.append(info_data)
    
    if info_buffer:
        stitch(info_buffer)


def count_missing_semantics():
    global title, scene, output_dir
    
    input_file = os.path.join(output_dir, f"{title}_{scene}.json")
    df = pd.read_json(input_file, orient='records', lines=True)
    
    output_content = ""
    
    for actor_type, group in df.groupby('actor_type'):
        count_with_empty_exec = group[group['exec'] == ''].shape[0]
        total = group.shape[0]
        percent_missing_exec = (count_with_empty_exec / total * 100) if total > 0 else 0
        output_content += (f"Actor Type: {actor_type}\n"
                           f"Count of actorIDs with missing exec: {count_with_empty_exec}\n"
                           f"Percentage with missing exec: {percent_missing_exec:.2f}%\n\n")
    
    for object_type, group in df.groupby('object'):
        count_with_empty_path = group[group['properties'] == ''].shape[0]
        total = group.shape[0]
        percent_missing_path = (count_with_empty_path / total * 100) if total > 0 else 0
        output_content += (f"Object Type: {object_type}\n"
                           f"Count of objectIDs with missing properties: {count_with_empty_path}\n"
                           f"Percentage with missing properties: {percent_missing_path:.2f}%\n\n")
    
    outfile = os.path.join(output_dir, f"{title}_{scene}_meta.txt")
    
    with open(outfile, "w") as file:
        file.write(output_content)

def main():
    global title, scene, cdm, output_dir, uid, uname
    directory = 'raw_gz_files'
    title = 'E3'
    scene = 'theia'
    output_dir = 'json_output_files'
    os.makedirs(output_dir, exist_ok=True)

    cdm = "18" if title == 'E3' else "20"

    extract_json_from_tar_gz(directory)
    prepare_id_files(directory)
    query_json(directory)
    count_missing_semantics()


if __name__ == '__main__':
    main()

In [ ]:
df = pd.read_json('json_output_files/E3_theia.json',lines=True)
df['timestampNanos'] = pd.to_datetime(df['timestampNanos'])

In [ ]:
df.drop('timestamp', axis=1, inplace=True)
df.rename(columns={'timestampNanos': 'timestamp'}, inplace=True)

In [ ]:
df.drop('exec', axis=1, inplace=True)
df.rename(columns={'cmdline': 'exec'}, inplace=True)

In [ ]:
train_df = df[df['timestamp'] < "2018-04-04 00:00"].copy(deep=True)
test_df = df[(df['timestamp'] > "2018-04-12 00:00") & (df['timestamp'] < "2018-04-12 23:00")].copy(deep=True)

In [ ]:
len(set(train_df['exec']))

In [ ]:
Train = True

In [ ]:
from pprint import pprint
import gzip
from sklearn.manifold import TSNE
import json
import copy
import os

In [ ]:
def add_node_properties(nodes, node_id, properties):
    if node_id not in nodes:
        nodes[node_id] = []
    nodes[node_id].extend(properties)

def update_edge_index(edges, edge_index, index):
    for src_id, dst_id in edges:
        src = index[src_id]
        dst = index[dst_id]
        edge_index[0].append(src)
        edge_index[1].append(dst)

def prepare_graph(df):
    nodes, labels, edges = {}, {}, []
    lblmap = {}
    
    dummies = {'SUBJECT_PROCESS': 0, 'FILE_OBJECT_FILE': 1, 'NetFlowObject': 2}
    
    for _, row in df.iterrows():
        action = row.get("action")
        raw_props = row.get("properties", {})
        
        if not raw_props:
            continue
    
        properties = [row.get("exec"), action]
        properties = properties + ([raw_props.get("cmdline")] if raw_props.get("cmdline") else [])
        properties = properties + ([raw_props.get("objectpath")] if raw_props.get("objectpath") else [])
        properties = properties + ([raw_props.get("remoteAddress")] if raw_props.get("remoteAddress") else [])
        properties = properties + ([raw_props.get('remotePort')] if raw_props.get('remotePort') else [])
        
        actor_id = row["actorID"]
        add_node_properties(nodes, actor_id, properties)
        labels[actor_id] = dummies[row['actor_type']]

        object_id = row["objectID"]
        add_node_properties(nodes, object_id, properties)
        labels[object_id] = dummies[row['object']]

        lblmap[actor_id] = row.get("exec")
        lblmap[object_id] = raw_props.get("objectpath") if raw_props.get("objectpath") else raw_props.get("remoteAddress", '')

        edges.append((actor_id, object_id))

    features, feat_labels, edge_index, index_map = [], [], [[], []], {}
    for node_id, props in nodes.items():
        features.append(props)
        feat_labels.append(labels[node_id])
        index_map[node_id] = len(features) - 1

    update_edge_index(edges, edge_index, index_map)

    return features, feat_labels, edge_index, list(index_map.keys()), lblmap

In [ ]:
from torch_geometric.nn import GCNConv
from torch_geometric.nn import SAGEConv, GATConv
import torch.nn.functional as F
import torch.nn as nn

class GCN(torch.nn.Module):
    def __init__(self,in_channel,out_channel):
        super().__init__()
        self.conv1 = SAGEConv(in_channel, 32, normalize=True)
        self.conv2 = SAGEConv(32, out_channel, normalize=True)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = x.relu()
        x = F.dropout(x, p=0.5, training=self.training)

        x = self.conv2(x, edge_index)
        return F.softmax(x, dim=1)

In [ ]:
def visualize(h, color):
    z = TSNE(n_components=2).fit_transform(h.detach().cpu().numpy())

    plt.figure(figsize=(10,10))
    plt.xticks([])
    plt.yticks([])

    plt.scatter(z[:, 0], z[:, 1], s=70, c=color, cmap="Set2")
    plt.show()

In [ ]:
from gensim.models.callbacks import CallbackAny2Vec
import gensim
from gensim.models import Word2Vec
from multiprocessing import Pool
from itertools import compress
from tqdm import tqdm
import time

class EpochSaver(CallbackAny2Vec):
    '''Callback to save model after each epoch.'''

    def __init__(self):
        self.epoch = 0

    def on_epoch_end(self, model):
        model.save('word2vec_theia_E3.model')
        self.epoch += 1

In [ ]:
class EpochLogger(CallbackAny2Vec):
    '''Callback to log information about training'''

    def __init__(self):
        self.epoch = 0

    def on_epoch_begin(self, model):
        print("Epoch #{} start".format(self.epoch))

    def on_epoch_end(self, model):
        print("Epoch #{} end".format(self.epoch))
        self.epoch += 1

In [ ]:
logger = EpochLogger()
saver = EpochSaver()

In [ ]:
if Train:
    train_df.sort_values(by='timestamp', ascending=True,inplace=True)

In [ ]:
from sklearn.utils import class_weight
import torch.nn.functional as F
from torch.nn import CrossEntropyLoss

model = GCN(30,3).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)

In [ ]:
phrases, labels, edges,mapp, _ = prepare_graph(train_df)
if Train:
    word2vec = Word2Vec(sentences=phrases, vector_size=30, window=5, min_count=1, workers=8,epochs=300,callbacks=[saver,logger])

In [ ]:
import math
import torch
import numpy as np
from gensim.models import Word2Vec

class PositionalEncoder:

    def __init__(self, d_model, max_len=100000):
        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        self.pe = torch.zeros(max_len, d_model)
        self.pe[:, 0::2] = torch.sin(position * div_term)
        self.pe[:, 1::2] = torch.cos(position * div_term)

    def embed(self, x):
        return x + self.pe[:x.size(0)]


def infer(document):
    word_embeddings = [w2vmodel.wv[word] for word in document if word in  w2vmodel.wv]
    
    if not word_embeddings:
        return np.zeros(30)

    output_embedding = torch.tensor(word_embeddings, dtype=torch.float)
    if len(document) < 100000:
        output_embedding = encoder.embed(output_embedding)

    output_embedding = output_embedding.detach().cpu().numpy()
    return np.mean(output_embedding, axis=0)

encoder = PositionalEncoder(30)
w2vmodel = Word2Vec.load("word2vec_theia_E3.model")

In [ ]:
import os
import json
import pickle
import time

import torch
import numpy as np
from torch.nn import CrossEntropyLoss
from torch_geometric.data import Data
from torch_geometric.loader import NeighborLoader
from torch_geometric import utils
from sklearn.utils import class_weight
    
model = GCN(30,3).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)

#phrases, labels, edges,mapp, _ = prepare_graph(train_df)

nodes = [infer(x) for x in phrases]
nodes = np.array(nodes)

criterion = CrossEntropyLoss()

graph = Data(
    x=torch.tensor(nodes, dtype=torch.float).to(device),
    y=torch.tensor(labels, dtype=torch.long).to(device),
    edge_index=torch.tensor(edges, dtype=torch.long).to(device)
)

for epochs in range(100):
    model.train()
    optimizer.zero_grad() 
    out = model(graph.x, graph.edge_index) 
    loss = criterion(out, graph.y) 
    loss.backward() 
    optimizer.step()      
    total_loss = loss.item() 
    print(total_loss)

# Save model checkpoint
torch.save(model.state_dict(), f'explain_lword2vec_gnn_theia_E3.pth')

In [ ]:
phrases, labels, edges, mapp, lbl = prepare_graph(test_df)

nodes = [infer(x) for x in phrases]
nodes = np.array(nodes)  

graph = Data(x=torch.tensor(nodes,dtype=torch.float).to(device),y=torch.tensor(labels,dtype=torch.long).to(device), edge_index=torch.tensor(edges,dtype=torch.long).to(device))

model = GCN(30,3).to(device)
model.load_state_dict(torch.load(f'explain_lword2vec_gnn_theia_E3.pth'))
model.eval()

In [ ]:
from torch_geometric.utils import remove_self_loops
graph.edge_index, _ = remove_self_loops(graph.edge_index)

In [ ]:
node_labels = [lbl[x] for x in mapp]
graph.node_labels = node_labels

In [ ]:
print(node_labels)

In [ ]:
temp_df = test_df[test_df['exec'] != '']
temp = pd.Series(temp_df['exec'].values, index=temp_df['actorID']).to_dict()

for k,v in temp.items():
    lbl[k] = v

In [ ]:
indices = []
for i in range(len(labels)):
    if labels[i] == 0 and lbl[mapp[i]] != '':
        indices.append(i)

In [ ]:
indices = []
for i in range(len(labels)):
    indices.append(i)
    
for i in range(len(indices)):
    node_idx = indices[i]
    if "cache" in lbl[mapp[node_idx]]:
        print(i, lbl[mapp[node_idx]])

In [ ]:
from torch_geometric.explain import Explainer, GNNExplainer, ModelConfig
import torch

for i in range(len(indices)):
    
    node_idx = indices[i]
    #print(lbl[mapp[node_idx]])
    
    explainer = Explainer(
        model=model,
        algorithm=GNNExplainer(epochs=100),
        explanation_type='model',
        node_mask_type='attributes',
        edge_mask_type='object',
        model_config=dict(
            mode='multiclass_classification',
            task_level='node',
            return_type='log_probs',
        ),
    )
    
    explanation = explainer(
        x=graph.x,
        edge_index=graph.edge_index,
        target=graph.y,
        index=node_idx,
    )

    # Extract k-hop subgraph and compute normalized edge_mask
    subset, edge_index, mapping, hard_edge_mask = k_hop_subgraph(
        node_idx, num_hops=2, edge_index=graph.edge_index, relabel_nodes=True
    )
    edge_mask = explanation.edge_mask[hard_edge_mask].cpu().detach().numpy()

    if edge_mask.sum() > 0:
        print(edge_mask.sum(), indices[i])

In [ ]:
from torch_geometric.explain import Explainer, GNNExplainer, ModelConfig
import torch

node_idx = 21000
print(lbl[mapp[node_idx]])

explainer = Explainer(
    model=model,
    algorithm=GNNExplainer(epochs=200),
    explanation_type='model',
    node_mask_type='attributes',
    edge_mask_type='object',
    model_config=dict(
        mode='multiclass_classification',
        task_level='node',
        return_type='log_probs',
    ),
)

explanation = explainer(
    x=graph.x,
    edge_index=graph.edge_index,
    target=graph.y,
    index=node_idx,
)

explanation.edge_mask.sum()

In [ ]:
node_labels = [lbl[x] for x in mapp]
graph.node_labels = node_labels

In [ ]:
from pyvis.network import Network
import networkx as nx
from torch_geometric.utils import k_hop_subgraph

# Extract k-hop subgraph and compute normalized edge_mask
subset, edge_index, mapping, hard_edge_mask = k_hop_subgraph(
    node_idx, num_hops=2, edge_index=graph.edge_index, relabel_nodes=True
)
edge_mask = explanation.edge_mask[hard_edge_mask].cpu().detach().numpy()
edge_mask = (edge_mask - edge_mask.min()) / (edge_mask.max() - edge_mask.min())

# Build a NetworkX graph from the subgraph edges
G = nx.Graph()
edge_list = edge_index.cpu().numpy().T
G.add_edges_from(edge_list)

# Retrieve the node labels for the subgraph
subset_indices = subset.cpu().numpy()
node_labels = [graph.node_labels[i] for i in subset_indices]

# Create a PyVis Network
net = Network(notebook=True, cdn_resources='in_line', width='100%', height='750px')

# Add nodes to the visualization
for i, node in enumerate(G.nodes()):
    node_int = int(node) 
    label = node_labels[i]
    short_label = label if len(label) <= 15 else label.split('\\')[-1]  # shorten long labels
    title = f"Node {subset_indices[i]}: {label}"
    color = 'red' if int(subset_indices[i]) == int(node_idx) else 'blue'
    net.add_node(node_int, label=short_label, title=title, color=color)

# Highlight edges above the threshold and dim others
threshold = 0.90
for i, (u, v) in enumerate(G.edges()):
    u_int, v_int = int(u), int(v)
    weight = float(edge_mask[i])
    
    # Choose edge color based on threshold
    color = 'red' if weight >= threshold else 'grey'
    
    # You can also make edges thinner/thicker based on importance by tweaking 'value' or adding 'width'
    net.add_edge(
        u_int, 
        v_int, 
        #value=weight, 
        title=f"Edge weight: {weight:.2f}", 
        color=color
    )

# Optional: Configure physics layout
net.set_options("""
var options = {
  "physics": {
    "forceAtlas2Based": {
      "gravitationalConstant": -50,
      "centralGravity": 0.01,
      "springLength": 100,
      "springConstant": 0.08
    },
    "maxVelocity": 50,
    "solver": "forceAtlas2Based",
    "timestep": 0.35,
    "stabilization": {"iterations": 150}
  }
}
""")

# Show the resulting visualization
net.show('graph_explanation.html')

In [ ]:
edge_mask.sum()